In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LogisticRegression, RidgeClassifier, RidgeClassifierCV, Lasso, LassoCV, ElasticNet, ElasticNetCV
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score, precision_score
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, VotingClassifier, StackingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve, auc
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.feature_selection import SelectFromModel

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
print(f"Dimensions des données d'entrainement: {train.shape}")
print(f"Dimensions des données d'évaluation: {test.shape}")

Dimensions des données d'entrainement: (200000, 45)
Dimensions des données d'évaluation: (63544, 44)


In [4]:
train.head()

,Num_Acc,an,int,agg,dep,atm_0,atm_1,lum_0,lum_1,lum_2,...,grave,catu_1,catu_2,age_1,age_2,age_3,catv,obsm,choc,Nb_Vehicules
0,201300019239,13,1,1,160,1,0,1,0,0,...,0,0,0,0,0,1,5,2,3,2
1,201600032158,16,0,0,670,1,0,1,0,0,...,1,1,0,0,0,1,4,2,1,2
2,201000032941,10,1,1,420,1,0,1,0,0,...,0,0,0,0,0,0,4,2,3,2
3,202000037509,20,0,1,49,1,0,1,0,0,...,0,0,0,0,0,1,5,2,0,2
4,202200040030,22,1,1,44,1,0,1,0,0,...,0,0,0,1,0,0,5,2,1,2


In [5]:
y = train.loc[:, "grave"]
X = train.loc[:, ~train.columns.isin(["Num_Acc", "grave"])]

test2 = test.iloc[:, 1:]

In [6]:
print(f"Dimensions des données d'entrainement: {X.shape}, {y.shape}")
print(f"Dimensions des données d'évaluation: {test2.shape}")

Dimensions des données d'entrainement: (200000, 43), (200000,)
Dimensions des données d'évaluation: (63544, 43)


In [7]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
print(f"Dimensions des données d'entrainement: {X_train.shape}, {y_train.shape}")
print(f"Dimensions des données de test: {X_val.shape}, {y_val.shape}")

Dimensions des données d'entrainement: (160000, 43), (160000,)
Dimensions des données de test: (40000, 43), (40000,)


# RandomForestClassifier

In [22]:
params = {
    "n_estimators": [100, 150, 200],
    "criterion": ["gini", "entropy"],
    "max_features": ["sqrt", "log2", 0.4, 0.5]
}

clf = RandomForestClassifier()
grid_search = GridSearchCV(clf, params, cv=5)
grid_search.fit(X_train, y_train)
grid_search.best_params_

{'criterion': 'entropy', 'max_features': 0.5, 'n_estimators': 200}

In [23]:
params_forest = grid_search.best_params_

In [120]:
clf = RandomForestClassifier(**{'criterion': 'entropy', 'max_features': 0.5, 'n_estimators': 500})
clf = clf.fit(X_train, y_train)
y_pred = clf.predict(X_val)
accuracy_score(y_val, y_pred)

0.731925

In [121]:
y_score = clf.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, y_score)

0.7933596219276662

# GradientBoostingClassifier

### Modèle soumis

In [26]:
parametres = {
    'n_estimators': [400, 500],
    'learning_rate': [0.01, 0.1],
    'max_depth': [5, 7]
}

gb_clf = GradientBoostingClassifier()

grid_search = GridSearchCV(estimator=gb_clf, param_grid=parametres, cv=3)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=GradientBoostingClassifier(),
             param_grid={'learning_rate': [0.01, 0.01, 0.1],
                         'max_depth': [3, 5, 7],
                         'n_estimators': [100, 150, 200]})

In [27]:
grid_search.best_params_

{'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}

In [28]:
params_boosting = grid_search.best_params_

In [9]:
clf = GradientBoostingClassifier(**{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500})
clf = clf.fit(X_train, y_train)
y_pred = clf.predict(X_val)
accuracy_score(y_val, y_pred)

0.751225

In [10]:
y_score = clf.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, y_score)

0.8209126389587527

In [11]:
clf = GradientBoostingClassifier(**{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500})

clf = clf.fit(X_train, y_train)

feature_importances = clf.feature_importances_

sorted_indices = np.argsort(feature_importances)[::-1]
sorted_feature_names = X_train.columns[sorted_indices]
sorted_feature_importances = feature_importances[sorted_indices]

# Afficher les noms de colonnes et les importances de caractéristiques triées
for name, importance in zip(sorted_feature_names, sorted_feature_importances):
    print(f"Feature {name}: Importance = {importance}")

Feature dep: Importance = 0.2408174825727231
Feature agg: Importance = 0.21117003109345933
Feature catr: Importance = 0.11077009908617583
Feature circ: Importance = 0.053487463629420105
Feature nbv: Importance = 0.048982522891143083
Feature an: Importance = 0.036452250216454055
Feature equipement: Importance = 0.03409692972348102
Feature choc: Importance = 0.03349080343879385
Feature Nb_Vehicules: Importance = 0.02945644998313356
Feature utilisation: Importance = 0.022366661198709045
Feature vma: Importance = 0.019616865250649304
Feature obsm: Importance = 0.019215373109426555
Feature catv: Importance = 0.018592815195484183
Feature age_3: Importance = 0.018368566624389918
Feature int: Importance = 0.0179638486993442
Feature lum_2: Importance = 0.010303300267336262
Feature catu_2: Importance = 0.009721660464436448
Feature plan_0: Importance = 0.007150833691052392
Feature catu_1: Importance = 0.0061577388517046975
Feature plan_1: Importance = 0.005555410854470333
Feature col_2: Importanc

In [17]:
# Sélectionner les fonctionnalités avec une importance supérieure à un seuil donné
sfm = SelectFromModel(clf, threshold=0.001)
sfm.fit(X_train, y_train)

# Transformer les données pour inclure uniquement les fonctionnalités sélectionnées
X_train2 = sfm.transform(X_train)
X_val2 = sfm.transform(X_val)

# Afficher les dimensions des données après sélection des fonctionnalités
print("Nombre de fonctionnalités avant la sélection :", X_train.shape[1])
print("Nombre de fonctionnalités après la sélection :", X_train2.shape[1])
print("Nombre de fonctionnalités de validation avant la sélection :", X_val.shape[1])
print("Nombre de fonctionnalités de validation après la sélection :", X_val2.shape[1])

Nombre de fonctionnalités avant la sélection : 43
Nombre de fonctionnalités après la sélection : 39
Nombre de fonctionnalités de validation avant la sélection : 43
Nombre de fonctionnalités de validation après la sélection : 39


In [18]:
clf = GradientBoostingClassifier(**{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500})
clf = clf.fit(X_train2, y_train)
y_pred = clf.predict(X_val2)
accuracy_score(y_val, y_pred)

0.75195

In [19]:
print(f"precision: {precision_score(y_val, y_pred)}")
print(f"F1 score: {f1_score(y_val, y_pred)}")
print(f"recall: {recall_score(y_val, y_pred)}")

precision: 0.7310474335790792
F1 score: 0.6793562564632886
recall: 0.6344923336955209


In [20]:
y_score = clf.predict_proba(X_val2)[:, 1]
roc_auc_score(y_val, y_score)

0.821234707578298

In [ ]:
probas = clf.predict_proba(test2)

probas_grave = probas[:, 1]

In [ ]:
rendu = pd.DataFrame({"Num_Acc": test.iloc[:, 0], "GRAVE": probas_grave})

In [ ]:
rendu.head()

In [ ]:
rendu.shape

In [ ]:
rendu.to_csv("rendu.csv", index=False)